<a href="https://colab.research.google.com/github/CesarChaMal/ollama/blob/main/ollama-webui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Linux uninstall ollama
!rm -rf *
!echo $OLLAMA_HOST
!export OLLAMA_HOST=http://localhost:11434
import os
os.environ['OLLAMA_HOST'] = 'http://localhost:11434'
!echo $OLLAMA_HOST
!command -v systemctl >/dev/null && ps aux | grep 'ollama' | grep -v grep | awk '{print $2}' | xargs -r kill
!rm /etc/systemd/system/ollama.service
!sudo rm $(which ollama)
!sudo rm -r /usr/share/ollama
!sudo userdel ollama

# Download and run the Ollama Linux install script
!curl https://ollama.ai/install.sh | sh

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
#!ps -ef | grep apt
#!ps -ef | grep dpkg
#!command -v systemctl >/dev/null && ps aux | grep 'apt' | grep -v grep | awk '{print $2}' | xargs -r kill
#!command -v systemctl >/dev/null && ps aux | grep 'dpkg' | grep -v grep | awk '{print $2}' | xargs -r kill
!sudo dpkg --configure -a
!sudo apt-get update && sudo apt-get install -y cuda-drivers
#!ollama start
!nohup ollama start &
!nohup ollama serve &
!ps aux | grep ollama
!journalctl -u ollama.service


In [ ]:
!rm -rf *
#!command -v systemctl
#!ps aux | grep '8080'
#!ps -ef | grep apt
#!ps -ef | grep dpkg
#!command -v systemctl >/dev/null && ps aux | grep 'apt' | grep -v grep | awk '{print $2}' | xargs -r kill
#!command -v systemctl >/dev/null && ps aux | grep 'dpkg' | grep -v grep | awk '{print $2}' | xargs -r kill
#!sudo dpkg --configure -a
!sudo apt-get install -y net-tools
!lsof -i :8000
!lsof -i :11434
!netstat -ltnp | grep :8000
!netstat -ltnp | grep :11434
!ps -aef
#!command -v systemctl >/dev/null && ps aux | grep '8000' | grep -v grep | awk '{print $2}' | xargs -r kill
#!lsof -i :8080 -t | xargs -r kill
#!sudo netstat -ltnp | grep ':8000' | awk '{print $7}' | cut -d'/' -f1 | xargs -r kill

!pip install aiohttp pyngrok
!pip show pyngrok
!pip install --upgrade pyngrok

import os
import asyncio
from aiohttp import ClientSession
from pyngrok import ngrok

# Set LD_LIBRARY_PATH so the system NVIDIA library becomes preferred
# over the built-in library. This is particularly important for
# Google Colab which installs older drivers
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

async def run(cmd):
    '''
    run is a helper function to run subcommands asynchronously.
    '''
    print('>>> starting', *cmd)
    p = await asyncio.subprocess.create_subprocess_exec(
        *cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )

    async def pipe(lines):
        async for line in lines:
            print(line.strip().decode('utf-8'))

    await asyncio.gather(
        pipe(p.stdout),
        pipe(p.stderr),
    )


#await asyncio.gather(
#    run(['ollama', 'serve']),
#    run(['ngrok', 'http', '--log', 'stderr', '11434']),
#)


async def run_ollama_commands():
    # Define the commands you want to run
    commands = [
        ['ollama', 'list'],
        ['ollama', 'run', 'mistral'],
        ['ollama', 'run', 'deepseek-coder'],
        ['ollama', 'run', 'llava']
    ]

    for cmd in commands:
        print('>>> Executing:', ' '.join(cmd))
        process = await asyncio.create_subprocess_exec(*cmd, stdout=asyncio.subprocess.PIPE,
                                                       stderr=asyncio.subprocess.PIPE)

        # Capture the output
        stdout, stderr = await process.communicate()
        if stdout:
            print(stdout.decode())
        if stderr:
            print(stderr.decode())


# Run the ollama commands
await run_ollama_commands()
#ngrok.disconnect(public_url.public_url)

# Setting Ollama Web UI
!curl -fsSL https://deb.nodesource.com/setup_16.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node -v
!npm -v

!git clone https://github.com/CesarChaMal/ollama-webui.git
!mv ollama-webui/* .

!cp -RPp example.env .env
!cat .env

!npm install vite
!npm install
!npm run build

!pip install uvicorn
!pip install --ignore-installed blinker
#!pip install --upgrade --force-reinstall blinker
!pip install kaleido
!pip install openai
!pip install cohere
!pip install tiktoken
!pip install -r backend/requirements.txt -U
#!./backend/start.sh
#!uvicorn backend.main:app --host 0.0.0.0 --port 8080
#!uvicorn backend.main:app --host 0.0.0.0 --port 8000 --forwarded-allow-ips '*'
#!uvicorn backend.main:app --reload --host 0.0.0.0 --port 8000 --forwarded-allow-ips '*'

#!nohup uvicorn backend.main:app --reload --host 0.0.0.0 --port 8000 --forwarded-allow-ips '*' > backend.log 2>&1 &

#!(uvicorn backend.main:app --reload --host 0.0.0.0 --port 8000 --forwarded-allow-ips '*' > backend.log 2>&1 &) && UVICORN_PID=$! && wait $UVICORN_PID

import subprocess
import requests
import time
import threading
import uvicorn
from fastapi import FastAPI
from pyngrok import ngrok

# Replace these with your actual keys
openai_api_key = "your_openai_api_key_here"
ngrok_authtoken = "your_ngrok_auth_token_here"

# Construct the sed commands
sed_openai = f"sed -i 's/^OPENAI_API_KEY=.*/OPENAI_API_KEY={openai_api_key}/' .env"
sed_ngrok = f"sed -i 's/^NGROK_AUTHTOKEN=.*/NGROK_AUTHTOKEN={ngrok_authtoken}/' .env"

# Execute the commands
subprocess.run(sed_openai, shell=True)
subprocess.run(sed_ngrok, shell=True)

from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv()

# Access the NGROK_AUTHTOKEN
ngrok_token = os.getenv('NGROK_AUTHTOKEN')
openai_key = os.getenv('OPENAI_API_KEY')

# Define a global event that signals the server is ready
server_ready = threading.Event()

# Function to execute shell commands and print their output
def run_shell_command(cmd):
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
    stdout, stderr = process.communicate()
    if stdout:
        print(stdout.decode())
    if stderr:
        print(stderr.decode())

# Function to check server readiness
def check_server_ready(port, retries=5, delay=2):
    """Check server readiness by making a GET request."""
    url = f'http://localhost:{port}/'
    for _ in range(retries):
        try:
            response = requests.get(url)
            if response.status_code == 200:
                print("Server is ready!")
                server_ready.set()
                return True
        except requests.ConnectionError:
            print("Server not ready, waiting...")
            time.sleep(delay)
    print("Server failed to start within the expected time.")
    return False

# Start Uvicorn server in the background
command = "nohup uvicorn backend.main:app --reload --host 0.0.0.0 --port 8000 --forwarded-allow-ips '*' > backend.log 2>&1 &"
subprocess.Popen(command, shell=True)

# Assuming the server is running on port 8000
if check_server_ready(8000):
    # Run your desired commands
    run_shell_command("cat backend.log")
    run_shell_command("lsof -i :11434")
    run_shell_command("lsof -i :8000")
    run_shell_command("netstat -ltnp | grep :11434")
    run_shell_command("netstat -ltnp | grep :8000")
    run_shell_command("ps -aef")
    run_shell_command("curl http://localhost:11434")
    run_shell_command("curl http://localhost:8000")

    # Set up ngrok and create a tunnel
    ngrok.set_auth_token(ngrok_token)

    public_url = ngrok.connect(8000, "http")
    print("Ngrok Tunnel URL:", public_url)

    tunnel_url = public_url.public_url  # Extract the actual URL string
    print("Ngrok Tunnel URL:", tunnel_url)

    # Optionally check ngrok log (if applicable)
    try:
        with open('/tmp/ngrok.log', 'r') as f:
            print(f.read())
    except FileNotFoundError:
        print("Ngrok log file not found.")


# Extract the actual URL from the ngrok object
actual_url = str(public_url).strip()

# Print the actual URL to verify
print("Actual URL:", actual_url)

!pip install Pygments
!curl http://localhost:11434 | pygmentize -l console
!curl http://0.0.0.0:8000 | pygmentize -l console

# Use the actual URL in the curl command
os.system(f"curl {actual_url} | pygmentize -l console")